# 03 — Train the mobile Adaptive Styles baseline
See `../TRAINING.md` for supervision, splits and limitations. Stages are disabled by default. Enable and run one stage at a time. Use this notebook's CUDA-enabled Python kernel; child commands use the same interpreter. Labels come from a pretrained aesthetic teacher and can be reviewed before student training.

In [ ]:
from pathlib import Path
import os, sys, subprocess

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'train.py').is_file() and (p / 'configs/baseline.json').is_file()), None)
if root is None:
    raise RuntimeError('Start Jupyter from research/adaptive-styles.')
print('Workspace:', root)
print('Python:', sys.executable)

def command(arguments, enabled=False):
    args = [sys.executable, '-u', str(root / 'train.py'), *map(str, arguments)]
    print('Command:', ' '.join(args))
    if not enabled:
        print('Disabled: enable the corresponding stage flag to run.')
        return
    environment = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    with subprocess.Popen(args, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=environment) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise RuntimeError('Stage failed; see output above. Do not continue until resolved.')
        except BaseException:
            process.terminate()
            process.wait()
            raise


In [ ]:
COUNT = 1000
EPOCHS = 20
BATCH_SIZE = 8
DATA = root / 'data/processed/baseline-v1'
RUN = root / 'outputs/baseline-v1'
REVIEW = root / 'outputs/label-review'
PREPARE = False
LABEL = False
REVIEW_LABELS = False
TRAIN = False
EXPORT = False
USE_REVIEW_OVERRIDES = False


## 1. Freeze working images and scene splits
Apple images are read from `data/Apple/`; downloaded references and input photos are read from `data/raw/`. Add known burst/crop groups in `data/metadata/groups.csv` before preparation. To change the frozen selection later, choose a new DATA and RUN path.

In [ ]:
if PREPARE and (DATA / 'manifest.json').exists():
    print('Keeping the existing frozen dataset:', DATA)
else:
    command(['prepare', '--data', DATA, '--max-images', COUNT], PREPARE)


## 2. Generate and review candidate preferences
The first run downloads verified teacher weights. Candidate scoring is the slow stage. Completed labels resume automatically. Review sheets include unchanged Standard and candidate IDs. Edit overrides.csv to select another candidate or set accept=0; these are local research annotations, not Apple metadata.

In [ ]:
command(['label', '--data', DATA, '--batch-size', BATCH_SIZE, '--device', 'cuda'], LABEL)
command(['review', '--data', DATA, '--output', REVIEW, '--limit', 50], REVIEW_LABELS)
print('Review HTML:', REVIEW / 'index.html')


## 3. Fine-tune MobileNetV3-Small
The first epoch trains heads; later epochs fine-tune the last feature blocks. AMP and gradient accumulation reduce memory. Reuse the same reviewed labels and settings when resuming. Validation reports label agreement, not human aesthetic quality. Leave the test set for final evaluation.

In [ ]:
arguments = ['train', '--data', DATA, '--run', RUN, '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--device', 'cuda']
if (RUN / 'last.pt').exists():
    arguments += ['--resume', RUN / 'last.pt']
if USE_REVIEW_OVERRIDES:
    arguments += ['--overrides', REVIEW / 'overrides.csv']
command(arguments, TRAIN)


## 4. Export an isolated research model
Export checks ONNX Runtime numerical parity and writes preprocessing/Style-ID metadata. It does not install the model into the released app. Check human preferences, real iPhone inputs and native Photos color/HDR behavior before integration.

In [ ]:
command(['export', '--checkpoint', RUN / 'best.pt', '--output', root / 'exports/baseline-v1'], EXPORT)
